In [ ]:
import pyspedas
import pytplot
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import matplotlib.cm as cm
# %matplotlib tk

import scipy
from scipy import interpolate,optimize
from scipy.optimize import curve_fit
from skimage.transform import probabilistic_hough_line

import helper
from helper import UTC_to_UNX
from helper import UNX_to_UTC
from helper import find_closest_index_dt

import math
from scipy.interpolate import interp1d
from numpy.linalg import LinAlgError

In [ ]:
from detection_helper import *

In [ ]:
times_arr, freq_arr, data_arr = return_arr('2023-08-19', '2023-08-20')

In [ ]:
freq_log, freq_log_exp, data_arr_log = convert_data_log(freq_arr, data_arr)

In [ ]:
bmap_row_mean = bmap_row_mean_loop_new(times_arr=times_arr, full_data=data_arr_log, min_duration=1)

plt.figure(figsize=(10, 6))
plt.imshow(1-bmap_row_mean.T, aspect='auto', origin='lower',cmap='gray')
plt.show()

In [ ]:
fig, ax = plt.subplots()
p = ax.pcolormesh(times_arr, freq_log_exp, data_arr_log.T, norm=LogNorm())
plt.colorbar(p, label='Power')

plt.xlabel("Time")
plt.ylabel("Frequency [Hz]")
# plt.yscale('log')
plt.title("Grouped Bursts")
# plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
def hough_angle(freq_change, freq_log, time_duration, times_arr):
    # times in second
    # times_arr in UTC
    target_freq = max(freq_log) - freq_change
    target_freq_index = 0
    while target_freq_index < len(freq_log) and freq_log[target_freq_index] < target_freq:
        target_freq_index += 1
    target_freq_index_reverse = len(freq_log) - target_freq_index

    times_arr_UNX = UTC_to_UNX(times_arr)
    target_time = times_arr_UNX[1] + time_duration
    target_time_index = 0
    while target_time_index < len(times_arr_UNX) and times_arr_UNX[target_time_index] < target_time:
        target_time_index += 1

    angle_rad = math.atan2(-target_freq_index_reverse, target_time_index)
    angle_deg = math.degrees(angle_rad) + 90

    return target_freq_index_reverse, target_time_index, angle_deg

freq_change_mean=14718120.082815735
duration_mean=2699.503105590062

target_freq_index_reverse, target_time_index, angle_deg = hough_angle(freq_change_mean, freq_log, duration_mean, times_arr)
target_freq_index_reverse, target_time_index, angle_deg

In [ ]:
bmap_row_mean.shape

In [ ]:
type(bmap_row_mean)

In [ ]:
bmap_row_mean_sliced = bmap_row_mean[:800,:]

plt.figure(figsize=(10, 6))
plt.imshow(1-bmap_row_mean_sliced.T, aspect='auto', origin='lower',cmap='gray')
plt.show()

In [ ]:
bmap_row_mean_low = bmap_row_mean[:800,10:40]

plt.figure(figsize=(10, 6))
plt.imshow(1-bmap_row_mean_low.T, aspect='auto', origin='lower',cmap='gray')
plt.show()

In [ ]:
bmap_row_mean_mid = bmap_row_mean[:800,40:80]

plt.figure(figsize=(10, 6))
plt.imshow(1-bmap_row_mean_mid.T, aspect='auto', origin='lower',cmap='gray')
plt.show()

In [ ]:
bmap_row_mean_high = bmap_row_mean[:800,80:]

plt.figure(figsize=(10, 6))
plt.imshow(1-bmap_row_mean_high.T, aspect='auto', origin='lower',cmap='gray')
plt.show()

In [ ]:
lines = hough_detect(bmap_row_mean_sliced, data_arr_log[:800,:], threshold=1, line_gap=50, line_length=100, theta=np.deg2rad(np.linspace(0, 10, 120)))

time_diff = 1000
freq_diff = 100000000
line_sets, line_sets_actual = line_grouping_new(times_arr, freq_log, lines, time_diff, freq_diff)
line_sets_actual

In [ ]:
lines

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
# p = ax.pcolormesh(times_arr, freq_log_exp, data_arr.T, norm=LogNorm())
# p = ax.imshow(1-bmap_row_mean_sliced.T, aspect='auto', origin='lower',cmap='gray')
# plt.colorbar(p, label='Power')

Z = 1 - bmap_row_mean_sliced.T
t = times_arr[:800]
f = freq_log_exp

p = ax.pcolormesh(t, f, Z, shading="auto", cmap="gray")
plt.colorbar(p, ax=ax, label="Binary")

for (x0, y0), (x1, y1) in lines:
    t0, t1 = times_arr[:800][y0], times_arr[:800][y1]
    f0, f1 = freq_log_exp[x0], freq_log_exp[x1]
    # print(t0, t1, f0, f1)
    ax.plot([t0, t1], [f0, f1], color='red')

plt.xlabel("Time")
plt.ylabel("Frequency [Hz]")
# plt.yscale('log')
plt.title("Grouped Bursts")
# plt.grid(True)
plt.tight_layout()
plt.show()